# Chapter 3 Practical 06: Cold Start, Sparsity, Clustering, and Temporal Dynamics

Learning objectives:
- Detect cold-start users and items.
- Discuss sparsity, popularity bias, and cold start.
- Cluster users to reduce neighbor search.
- Add time-decay weights to recent interactions.
- Complete three short Chapter 3 challenges.

Slide connection: limits of memory-based CF, practical mitigations, clustering for scalability, and temporal dynamics.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIRS = [
    Path("data"),
    Path("../data"),
    Path("chapter_03_collaborative_filtering/data"),
]
GITHUB_DATA_URL = "https://raw.githubusercontent.com/MehrdadJalali-AI/RecommenderSystems/main/chapter_03_collaborative_filtering/data"

def read_chapter3_csv(filename):
    for data_dir in DATA_DIRS:
        csv_path = data_dir / filename
        if csv_path.exists():
            print(f"Loaded {filename} from {csv_path}")
            return pd.read_csv(csv_path)
    url = f"{GITHUB_DATA_URL}/{filename}"
    print(f"Local file not found. Loading {filename} from GitHub raw URL.")
    return pd.read_csv(url)

ratings = read_chapter3_csv("ratings_chapter3.csv")
movies = read_chapter3_csv("movies_chapter3.csv")
ratings_named = ratings.merge(movies, on="movie_id", how="left")
rating_matrix = ratings_named.pivot_table(index="user_id", columns="title", values="rating")
rating_matrix


Cold-start users and cold-start items have too few interactions for reliable memory-based CF.


In [ ]:
user_counts = ratings_named.groupby("user_id").size().rename("ratings_count")
item_counts = ratings_named.groupby("title").size().rename("ratings_count")

print("Cold-start-like users:")
display(user_counts[user_counts <= 2])
print("Cold-start-like items:")
display(item_counts[item_counts <= 2])


Sparsity means that most user-item pairs are unknown. Popularity bias means popular items get more evidence and may be recommended more often.


In [ ]:
n_users, n_items = rating_matrix.shape
known_ratings = rating_matrix.notna().sum().sum()
sparsity = 1 - known_ratings / (n_users * n_items)

popularity = (
    ratings_named.groupby("title")["rating"]
    .agg(ratings_count="count", mean_rating="mean")
    .sort_values(["ratings_count", "mean_rating"], ascending=False)
)

print(f"Matrix sparsity: {sparsity:.1%}")
popularity.head(5).round(2)


One scalability mitigation is to compare the target user only with users in the same cluster. Here we fill missing values with item means only for clustering, not as real ratings.


In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

filled = rating_matrix.apply(lambda col: col.fillna(col.mean()), axis=0)
scaled = StandardScaler().fit_transform(filled)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = pd.Series(kmeans.fit_predict(scaled), index=rating_matrix.index, name="cluster")
clusters.to_frame().sort_values("cluster")


In [ ]:
target_user = "Karen"
target_cluster = clusters.loc[target_user]
candidate_neighbors = clusters[clusters.eq(target_cluster)].index.drop(target_user)
print(f"Compare Karen only with users in cluster {target_cluster}: {candidate_neighbors.tolist()}")


Temporal dynamics give more weight to recent interactions. A larger decay value makes older ratings fade more quickly.


In [ ]:
decay_lambda = 0.01
ratings_named["time_weight"] = np.exp(-decay_lambda * ratings_named["days_ago"])
ratings_named["weighted_rating"] = ratings_named["rating"] * ratings_named["time_weight"]

ratings_named[["user_id", "title", "rating", "days_ago", "time_weight", "weighted_rating"]].sort_values("days_ago").head(10).round(3)


In [ ]:
recent_profile = (
    ratings_named.groupby("title")
    .apply(lambda g: np.average(g["rating"], weights=g["time_weight"]))
    .rename("time_weighted_mean")
    .sort_values(ascending=False)
)
recent_profile.head(5).round(2)


# Challenge 1 - Change the Neighborhood Size

## Goal

Investigate how the number of neighbors influences a prediction.

1. Run the existing prediction using the current value of `k_neighbors`.
2. Change the value of `k_neighbors`, for example:

```python
k_neighbors = 2
```

and then:

```python
k_neighbors = 5
```

3. Rerun the recommendation or rating-prediction code.
4. Compare the predicted rating or recommendation list.


In [ ]:
# Challenge 1: Write or modify your code here


> **Your observations:**
> How did changing `k` affect the prediction or recommendation results?
> Why can using too few or too many neighbors influence the result?


# Challenge 2 - Compare Cosine and Pearson Similarity

## Goal

Investigate whether the selected similarity measure changes the neighbors and recommendations.

1. Run User-User CF using cosine similarity.
2. Change the code to use Pearson correlation.
3. Compare:

- the top-k neighbors,
- similarity values,
- the predicted rating or recommendation list.


In [ ]:
# Challenge 2: Modify the similarity method and rerun the recommender


> **Your observations:**
> Did cosine similarity and Pearson correlation select the same neighbors?
> Which method appeared more suitable for these users, and why?


# Challenge 3 - Concept Check: Cold Start

This challenge does not require programming.

> A new user joins a movie platform but has not rated, liked, or watched any movies.
> At the same time, a newly released movie has not yet received any interactions.

Answer:

1. Why can standard memory-based collaborative filtering not provide reliable personalized recommendations in these two cases?
2. Suggest one practical solution for the new user.
3. Suggest one practical solution for the new item.

> **Your explanation:**
>
> New-user problem:
>
> ................................................................................
>
> Suggested solution:
>
> ................................................................................
>
> New-item problem:
>
> ................................................................................
>
> Suggested solution:
>
> ................................................................................
